In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Código 7: Modelo Random Forest para predicción horaria de O3 a 72h.
Entrada: datasets 2D generados en código 6 (ml_2d) tanto por estación como global.
Salida: modelos entrenados, métricas, gráficas y archivos de resultados.
"""

import os
import json
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# ============================================================================
# CONFIGURACIÓN
# ============================================================================

BASE_DIR = os.path.expanduser("~/Documents/GitHub/TFGFinal/Datos_TFG_outliers")
DATA_DIR = os.path.join(BASE_DIR, "windows_partitioned")
OUTPUT_DIR = os.path.join(BASE_DIR, "models", "random_forest")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Parámetros
WINDOW_OUT = 72
RANDOM_STATE = 42
N_JOBS = -1  # usar todos los núcleos

# Hiperparámetros para búsqueda (GridSearch)
RF_PARAM_GRID = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Métricas a calcular
def willmott_index(y_true, y_pred):
    """Índice de acuerdo de Willmott (0-1, 1 es perfecto)."""
    numer = np.sum((y_true - y_pred) ** 2)
    denom = np.sum((np.abs(y_pred - y_true.mean()) + np.abs(y_true - y_true.mean())) ** 2)
    return 1 - numer / denom if denom != 0 else np.nan

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error, evitando división por cero."""
    mask = y_true != 0
    if not mask.any():
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def compute_metrics(y_true, y_pred):
    """Calcula todas las métricas."""
    return {
        'r2': r2_score(y_true, y_pred),
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': np.sqrt(mean_squared_error(y_true, y_pred)),
        'mape': mape(y_true, y_pred),
        'willmott': willmott_index(y_true, y_pred)
    }

# ============================================================================
# FUNCIONES AUXILIARES
# ============================================================================

def plot_predictions(y_true, y_pred, horizons, save_path, title):
    """
    Grafica predicciones vs reales para horizontes específicos.
    y_true: array (n_samples, 72)
    y_pred: array (n_samples, 72)
    horizons: lista de índices de hora (ej. [23, 47, 71] para 24,48,72)
    """
    n_plots = len(horizons)
    fig, axes = plt.subplots(1, n_plots, figsize=(5*n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    for ax, h in zip(axes, horizons):
        ax.scatter(y_true[:, h], y_pred[:, h], alpha=0.3, s=10)
        ax.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', lw=1)
        ax.set_xlabel('Real O3 (µg/m³)')
        ax.set_ylabel('Predicho O3 (µg/m³)')
        ax.set_title(f'Horizonte {h+1}h')
        ax.grid(True, alpha=0.3)
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close()

def train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test, 
                          station_name, output_subdir, is_global=False):
    """
    Entrena Random Forest con búsqueda de hiperparámetros (usando validación en train con TimeSeriesSplit)
    y evalúa en test.
    """
    print(f"\n--- Entrenando Random Forest para {station_name} ---")

    # 1. Búsqueda de hiperparámetros con validación cruzada temporal
    tscv = TimeSeriesSplit(n_splits=3)  # 3 splits para series temporales
    rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=N_JOBS)
    grid_search = GridSearchCV(
        estimator=rf,
        param_grid=RF_PARAM_GRID,
        cv=tscv,
        scoring='neg_mean_squared_error',
        n_jobs=N_JOBS,
        verbose=1
    )
    print("  Buscando mejores hiperparámetros...")
    grid_search.fit(X_train, y_train)  # y_train es (n_samples, 72) - multioutput

    best_model = grid_search.best_estimator_
    best_params = grid_search.best_params_
    print(f"  Mejores parámetros: {best_params}")

    # 2. Evaluación en validación (opcional, podemos ver score)
    val_pred = best_model.predict(X_val)
    val_metrics = compute_metrics(y_val.ravel(), val_pred.ravel())  # aplanar para métricas globales
    print(f"  Métricas en validación (sobre todas las horas): R2={val_metrics['r2']:.3f}, MAE={val_metrics['mae']:.2f}")

    # 3. Evaluación en test
    test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test.ravel(), test_pred.ravel())
    print(f"  Métricas en test: R2={test_metrics['r2']:.3f}, MAE={test_metrics['mae']:.2f}, RMSE={test_metrics['rmse']:.2f}, MAPE={test_metrics['mape']:.2f}%, Willmott={test_metrics['willmott']:.3f}")

    # 4. Guardar modelo y resultados
    model_path = os.path.join(output_subdir, "model.pkl")
    with open(model_path, 'wb') as f:
        pickle.dump(best_model, f)

    # Guardar parámetros y métricas
    results = {
        'best_params': best_params,
        'validation_metrics': val_metrics,
        'test_metrics': test_metrics,
        'n_train': len(X_train),
        'n_val': len(X_val),
        'n_test': len(X_test)
    }
    with open(os.path.join(output_subdir, "results.json"), 'w') as f:
        json.dump(results, f, indent=2)

    # 5. Gráficas para horizontes específicos (24, 48, 72)
    horizons = [23, 47, 71]  # índices 0-based: 24ª hora -> índice 23, etc.
    plot_predictions(y_test, test_pred, horizons,
                     save_path=os.path.join(output_subdir, "test_scatter.png"),
                     title=f"Random Forest - {station_name} - Test")

    # También guardar predicciones completas para análisis posteriores
    np.save(os.path.join(output_subdir, "test_pred.npy"), test_pred)
    np.save(os.path.join(output_subdir, "test_true.npy"), y_test)

    return best_model, test_metrics

# ============================================================================
# PROCESAMIENTO POR ESTACIÓN (INDIVIDUAL)
# ============================================================================

def process_by_station():
    print("\n" + "="*50)
    print("PROCESANDO MODELOS POR ESTACIÓN")
    print("="*50)

    # Ruta a los datos 2D por estación
    ml_2d_dir = os.path.join(DATA_DIR, "by_station", "ml", "ml_2d")
    if not os.path.exists(ml_2d_dir):
        print(f"No existe la carpeta: {ml_2d_dir}")
        return

    stations = [d for d in os.listdir(ml_2d_dir) if os.path.isdir(os.path.join(ml_2d_dir, d))]
    if not stations:
        print("No se encontraron estaciones con datos 2D.")
        return

    for station in stations:
        station_dir = os.path.join(ml_2d_dir, station)
        # Cargar datos
        X_train = np.load(os.path.join(station_dir, "train_X.npy"))
        y_train = np.load(os.path.join(station_dir, "train_y.npy"))
        X_val = np.load(os.path.join(station_dir, "val_X.npy"))
        y_val = np.load(os.path.join(station_dir, "val_y.npy"))
        X_test = np.load(os.path.join(station_dir, "test_X.npy"))
        y_test = np.load(os.path.join(station_dir, "test_y.npy"))

        # Crear subcarpeta de salida para esta estación
        out_subdir = os.path.join(OUTPUT_DIR, "by_station", station)
        os.makedirs(out_subdir, exist_ok=True)

        # Entrenar y evaluar
        try:
            train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                                  station, out_subdir, is_global=False)
        except Exception as e:
            print(f"Error procesando {station}: {e}")
            continue

# ============================================================================
# PROCESAMIENTO GLOBAL
# ============================================================================

def process_global():
    print("\n" + "="*50)
    print("PROCESANDO MODELO GLOBAL")
    print("="*50)

    global_ml_2d = os.path.join(DATA_DIR, "global", "ml", "ml_2d")
    if not os.path.exists(global_ml_2d):
        print(f"No existe la carpeta: {global_ml_2d}")
        return

    # Verificar que existen los archivos
    required = ["train_X.npy", "train_y.npy", "val_X.npy", "val_y.npy", "test_X.npy", "test_y.npy"]
    if not all(os.path.exists(os.path.join(global_ml_2d, f)) for f in required):
        print("Faltan archivos en la carpeta global ml_2d.")
        return

    X_train = np.load(os.path.join(global_ml_2d, "train_X.npy"))
    y_train = np.load(os.path.join(global_ml_2d, "train_y.npy"))
    X_val = np.load(os.path.join(global_ml_2d, "val_X.npy"))
    y_val = np.load(os.path.join(global_ml_2d, "val_y.npy"))
    X_test = np.load(os.path.join(global_ml_2d, "test_X.npy"))
    y_test = np.load(os.path.join(global_ml_2d, "test_y.npy"))

    # Carpeta de salida para modelo global
    out_subdir = os.path.join(OUTPUT_DIR, "global")
    os.makedirs(out_subdir, exist_ok=True)

    train_and_evaluate_rf(X_train, y_train, X_val, X_test, y_val, y_test,
                          "GLOBAL", out_subdir, is_global=True)

# ============================================================================
# RESUMEN FINAL
# ============================================================================

def generate_summary():
    """Crea un resumen de todas las métricas en un CSV."""
    summary = []
    # Por estación
    by_station_dir = os.path.join(OUTPUT_DIR, "by_station")
    if os.path.exists(by_station_dir):
        for station in os.listdir(by_station_dir):
            res_file = os.path.join(by_station_dir, station, "results.json")
            if os.path.exists(res_file):
                with open(res_file, 'r') as f:
                    data = json.load(f)
                summary.append({
                    'station': station,
                    'type': 'by_station',
                    **data['test_metrics']
                })
    # Global
    global_res = os.path.join(OUTPUT_DIR, "global", "results.json")
    if os.path.exists(global_res):
        with open(global_res, 'r') as f:
            data = json.load(f)
        summary.append({
            'station': 'GLOBAL',
            'type': 'global',
            **data['test_metrics']
        })

    if summary:
        df = pd.DataFrame(summary)
        df.to_csv(os.path.join(OUTPUT_DIR, "summary_metrics.csv"), index=False)
        print("\nResumen guardado en:", os.path.join(OUTPUT_DIR, "summary_metrics.csv"))

# ============================================================================
# EJECUCIÓN PRINCIPAL
# ============================================================================

if __name__ == "__main__":
    print("="*60)
    print("RANDOM FOREST - ENTRENAMIENTO Y EVALUACIÓN")
    print("="*60)

    process_by_station()
    process_global()
    generate_summary()

    print("\nProceso completado. Revise la carpeta:")
    print(f"  {OUTPUT_DIR}")